Cleans raw 990 filings, engineers derived features, and writes two Parquet outputs:
- `filings_clean.parquet` — cleaned, human-readable (for UI display)
- `feature_matrix.parquet` — normalized ML-ready matrix (for XGBoost + FAISS)

In [1]:
# Data Audit
import pandas as pd

df = pd.read_parquet("../../data/processed/filings.parquet")

# Full null rates across all columns
print(df.isnull().mean().sort_values(ascending=False).to_string())

print()

# Value ranges on numeric columns
print(df[["totrevenue","totfuncexpns","totassetsend","totliabend",
          "compnsatncurrofcr","othrsalwages","payrolltx",
          "invstmntinc","totnetassetend","totprgmrevnue",
          "miscrevtot11e","pct_compnsatncurrofcr"]].describe().to_string())

totemployee              1.0000
org_subseccd             1.0000
totvoluntrs              1.0000
txblsalesprpty           1.0000
fundsrcvd                1.0000
prgmservrev              1.0000
grscontribs              1.0000
org_state                0.0319
totassetsend             0.0000
totrevenue               0.0000
totfuncexpns             0.0000
tax_prd_yr               0.0000
tax_prd                  0.0000
ein                      0.0000
formtype                 0.0000
grsrntsreal              0.0000
grsrntsprsnl             0.0000
rntlexpnsreal            0.0000
pct_compnsatncurrofcr    0.0000
invstmntinc              0.0000
txexmptbndsproceeds      0.0000
royaltsinc               0.0000
totliabend               0.0000
compnsatncurrofcr        0.0000
miscrevtot11e            0.0000
netincsales              0.0000
lessdirfndrsng           0.0000
rntlexpnsprsnl           0.0000
profndraising            0.0000
payrolltx                0.0000
othrsalwages             0.0000
nonpfrea

Imports & Load

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../data/processed")
RAW_PATH = DATA_DIR / "filings.parquet"

df = pd.read_parquet(RAW_PATH)

print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)

Loaded: 4232 rows, 39 columns


,ein,tax_prd,tax_prd_yr,formtype,totrevenue,totfuncexpns,totassetsend,totliabend,pct_compnsatncurrofcr,prgmservrev,...,nonpfrea,totemployee,totvoluntrs,org_name,org_address,org_city,org_state,org_zipcode,org_ntee_code,org_subseccd
0,010238552,202309,2023,0,3690471384,3513572710,3803040949,1638557025,0.0,NaN,...,3,NaN,NaN,A Mainehealth Hcsr,22 BRAMHALL ST,Portland,ME,04102-3134,E220,None
1,010238552,202209,2022,0,3440262667,3330800641,3765508907,1726178453,0.0,NaN,...,3,NaN,NaN,A Mainehealth Hcsr,22 BRAMHALL ST,Portland,ME,04102-3134,E220,None
2,010238552,202109,2021,0,3168653743,2844844262,4099747289,1866500792,0.0,NaN,...,03,NaN,NaN,A Mainehealth Hcsr,22 BRAMHALL ST,Portland,ME,04102-3134,E220,None


Drop Dead Columns
These were confirmed 100% null or zero-variance in the audit.

In [3]:
DEAD_COLUMNS = [ 
    "totemployee",        
    "totvoluntrs",        
    "txblsalesprpty",     
    "fundsrcvd",          
    "prgmservrev",        
    "grscontribs",        
    "org_subseccd",       
    "pct_compnsatncurrofcr",  
    "formtype",           # always 0 (filtered to full 990 only)
]

df = df.drop(columns=DEAD_COLUMNS)

print(f"After drop: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Remaining columns: {df.columns.tolist()}")

After drop: 4232 rows, 30 columns
Remaining columns: ['ein', 'tax_prd', 'tax_prd_yr', 'totrevenue', 'totfuncexpns', 'totassetsend', 'totliabend', 'invstmntinc', 'txexmptbndsproceeds', 'royaltsinc', 'grsrntsreal', 'grsrntsprsnl', 'rntlexpnsreal', 'rntlexpnsprsnl', 'lessdirfndrsng', 'netincsales', 'miscrevtot11e', 'compnsatncurrofcr', 'othrsalwages', 'payrolltx', 'profndraising', 'totprgmrevnue', 'totnetassetend', 'nonpfrea', 'org_name', 'org_address', 'org_city', 'org_state', 'org_zipcode', 'org_ntee_code']


Sort & Deduplicate
- Sort by EIN + year descending so each org's most recent filing is first.
- Drop exact duplicate EIN + tax period combinations if any crept in.

In [4]:
df = df.sort_values(["ein", "tax_prd_yr"], ascending=[True, False]).reset_index(drop=True)

before = len(df)
df = df.drop_duplicates(subset=["ein", "tax_prd"])
after = len(df)

print(f"Duplicates dropped: {before - after}")
print(f"Clean rows: {after}")

Duplicates dropped: 0
Clean rows: 4232


Clip Extreme Outliers
- `miscrevtot11e` has a min of -3.2B which would distort ratios.
- We clip financial columns at the 1st and 99th percentile to keep the
- feature distributions useful without deleting real data.

In [5]:
CLIP_COLS = [
    "totrevenue", "totfuncexpns", "totassetsend", "totliabend",
    "compnsatncurrofcr", "othrsalwages", "payrolltx",
    "invstmntinc", "totnetassetend", "totprgmrevnue",
    "miscrevtot11e",
]

for col in CLIP_COLS:
    low  = df[col].quantile(0.01)
    high = df[col].quantile(0.99)
    df[col] = df[col].clip(lower=low, upper=high)

print("Clipping complete. New ranges:")
df[CLIP_COLS].describe().loc[["min", "max"]].T

Clipping complete. New ranges:


,min,max
totrevenue,3.896367e+06,1.240255e+10
totfuncexpns,3.736779e+06,1.115074e+10
totassetsend,3.401807e+06,3.472304e+10
totliabend,0.000000e+00,1.149068e+10
compnsatncurrofcr,0.000000e+00,6.551054e+07
othrsalwages,0.000000e+00,3.946637e+09
payrolltx,0.000000e+00,2.501717e+08
invstmntinc,-1.455568e+04,4.491840e+08
totnetassetend,-1.945809e+08,2.693899e+10
totprgmrevnue,0.000000e+00,1.021250e+10


Engineer Derived Features
- These six ratios are the actual ML signals: not the raw dollar amounts.
- Raw amounts are biased by org size; ratios are size-agnostic.

| Feature | Formula | What it signals |
|---|---|---|
| `program_expense_ratio` | totprgmrevnue / totfuncexpns | How much spending goes to mission vs overhead |
| `admin_overhead_ratio` | (totfuncexpns - totprgmrevnue) / totrevenue | Admin bloat relative to revenue |
| `net_asset_margin` | totnetassetend / totrevenue | Financial cushion / reserve strength |
| `labor_cost_ratio` | (compnsatncurrofcr + othrsalwages + payrolltx) / totrevenue | Payroll burden |
| `solvency_ratio` | totassetsend / totliabend | Can they pay their debts? |
| `revenue_yoy_growth` | (rev_t - rev_t-1) / abs(rev_t-1) | Year-over-year revenue momentum |

In [6]:
# Safe division helper — returns NaN instead of inf when denominator is 0
def safe_div(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator.where(denominator != 0, other=np.nan) / denominator.replace(0, np.nan)

# --- Ratio features ---
df["program_expense_ratio"] = safe_div(df["totprgmrevnue"], df["totfuncexpns"])
df["admin_overhead_ratio"]  = safe_div(df["totfuncexpns"] - df["totprgmrevnue"], df["totrevenue"])
df["net_asset_margin"]      = safe_div(df["totnetassetend"], df["totrevenue"])
df["labor_cost_ratio"]      = safe_div(
    df["compnsatncurrofcr"] + df["othrsalwages"] + df["payrolltx"],
    df["totrevenue"]
)
df["solvency_ratio"]        = safe_div(df["totassetsend"], df["totliabend"])

# --- YoY revenue growth (per EIN, sorted by year) ---
# shift(1) within each EIN group gives the previous year's revenue
df["revenue_yoy_growth"] = df.groupby("ein")["totrevenue"].transform(
    lambda x: x.sort_values(ascending=True).pct_change()
)

ENGINEERED = [
    "program_expense_ratio",
    "admin_overhead_ratio",
    "net_asset_margin",
    "labor_cost_ratio",
    "solvency_ratio",
    "revenue_yoy_growth",
]

print("Engineered feature distributions:")
df[ENGINEERED].describe().T

Engineered feature distributions:


,count,mean,std,min,25%,50%,75%,max
program_expense_ratio,4232.0,0.657144,1.394070,0.000000,0.057870,0.836742,0.989536,6.291702e+01
admin_overhead_ratio,4232.0,0.474227,7.224720,-0.726004,0.009910,0.148206,0.664734,4.653045e+02
net_asset_margin,4232.0,2.589632,45.403423,-7.172712,0.194464,0.694896,1.540630,2.598526e+03
labor_cost_ratio,4232.0,0.292091,0.200353,0.000000,0.088231,0.336339,0.417904,3.580010e+00
solvency_ratio,4187.0,9490.808059,399516.384270,0.095837,1.522442,2.351743,4.102858,2.225054e+07
revenue_yoy_growth,3888.0,0.554747,13.872841,0.000000,0.024468,0.059184,0.116483,6.855683e+02


In [7]:
# Clip engineered features to financially sensible hard caps
HARD_CAPS = {
    "program_expense_ratio": (0.0,  1.5),
    "admin_overhead_ratio":  (-1.0, 2.0),
    "net_asset_margin":      (-5.0, 10.0),
    "labor_cost_ratio":      (0.0,  1.5),
    "solvency_ratio":        (0.0,  20.0),
    "revenue_yoy_growth":    (-1.0, 2.0),
}

for col, (low, high) in HARD_CAPS.items():
    pct_capped = (~df[col].between(low, high)).mean()
    df[col] = df[col].clip(lower=low, upper=high)
    print(f"{col:30s}  {pct_capped*100:.1f}% of rows capped")

print("\nPost-cap distributions:")
df[ENGINEERED].describe().loc[["min", "mean", "max"]].T

program_expense_ratio           0.0% of rows capped
admin_overhead_ratio            0.5% of rows capped
net_asset_margin                1.4% of rows capped
labor_cost_ratio                0.0% of rows capped
solvency_ratio                  8.8% of rows capped
revenue_yoy_growth              9.0% of rows capped

Post-cap distributions:


,min,mean,max
program_expense_ratio,0.000000,0.628689,1.5
admin_overhead_ratio,-0.726004,0.333352,2.0
net_asset_margin,-5.000000,1.239557,10.0
labor_cost_ratio,0.000000,0.291600,1.5
solvency_ratio,0.095837,4.362237,20.0
revenue_yoy_growth,0.000000,0.120668,2.0


Check for nulls and inf values introduced by division.

In [8]:
print("Null rates in engineered features:")
print(df[ENGINEERED].isnull().mean())

print("\nInf values:")
print(np.isinf(df[ENGINEERED].select_dtypes(include=np.number)).sum())

# Replace any inf that slipped through with NaN
df[ENGINEERED] = df[ENGINEERED].replace([np.inf, -np.inf], np.nan)

# Fill remaining NaNs in engineered features with the column median
for col in ENGINEERED:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print("\nNull rates after fill:")
print(df[ENGINEERED].isnull().mean())

Null rates in engineered features:
program_expense_ratio    0.000000
admin_overhead_ratio     0.000000
net_asset_margin         0.000000
labor_cost_ratio         0.000000
solvency_ratio           0.010633
revenue_yoy_growth       0.081285
dtype: float64

Inf values:
program_expense_ratio    0
admin_overhead_ratio     0
net_asset_margin         0
labor_cost_ratio         0
solvency_ratio           0
revenue_yoy_growth       0
dtype: int64

Null rates after fill:
program_expense_ratio    0.0
admin_overhead_ratio     0.0
net_asset_margin         0.0
labor_cost_ratio         0.0
solvency_ratio           0.0
revenue_yoy_growth       0.0
dtype: float64


Write filings_clean.parquet
- Human-readable cleaned data for the UI layer.
- Contains all columns including org metadata and raw financials.

In [9]:
CLEAN_PATH = DATA_DIR / "filings_clean.parquet"
df.to_parquet(CLEAN_PATH, index=False, engine="pyarrow")

print(f"filings_clean.parquet written: {df.shape[0]} rows, {df.shape[1]} cols")
print(f"Path: {CLEAN_PATH.resolve()}")

filings_clean.parquet written: 4232 rows, 36 cols
Path: C:\Users\Edrill-LT\Documents\Projects\Atlas990\backend\app\data\processed\filings_clean.parquet


Build Feature Matrix - Isolate only the engineered features + EIN identifier.

In [10]:
FEATURE_COLS = ENGINEERED  # the 6 derived ratios

# Take the most recent filing per EIN only
# Each org gets one vector — their latest financial fingerprint
latest = df.sort_values("tax_prd_yr", ascending=False).drop_duplicates(subset="ein").copy()

feature_matrix = latest[["ein", "org_name", "org_state", "org_ntee_code", "tax_prd_yr"] + FEATURE_COLS].copy()

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"One row per org: {feature_matrix['ein'].nunique() == len(feature_matrix)}")
feature_matrix.head(5)

Feature matrix shape: (344, 11)
One row per org: True


,ein,org_name,org_state,org_ntee_code,tax_prd_yr,program_expense_ratio,admin_overhead_ratio,net_asset_margin,labor_cost_ratio,solvency_ratio,revenue_yoy_growth
191,042632526,Elderhostel Inc,MA,N500,2024,1.046311,-0.041534,0.300270,0.102868,1.623510,0.085408
218,042775991,Metropolitan Boston Housing Partnership Inc,MA,L20,2024,0.984167,0.015566,0.058113,0.039607,1.535670,0.324916
1600,341747398,American Endowment Foundation,OH,T31,2023,0.000000,0.967544,5.393625,0.008035,20.000000,0.163413
0,010238552,A Mainehealth Hcsr,ME,E220,2023,0.867272,0.126366,0.586506,0.428569,2.320970,0.072730
972,223498690,Springpoint Senior Living Inc,NJ,P750,2023,0.853741,0.156922,-0.531689,0.345149,0.861733,0.114271


Normalize with StandardScaler
- StandardScaler transforms each feature to mean=0, std=1.
- This is required before FAISS — without it, solvency_ratio (which can be 50+) would completely dominate the distance calculations over ratios like program_expense_ratio (which lives between 0 and 1).

In [11]:
scaler = StandardScaler()

feature_matrix[FEATURE_COLS] = scaler.fit_transform(feature_matrix[FEATURE_COLS])

print("Scaled feature distributions (should be mean≈0, std≈1):")
feature_matrix[FEATURE_COLS].describe().loc[["mean", "std"]].T

Scaled feature distributions (should be mean≈0, std≈1):


,mean,std
program_expense_ratio,8.520316e-17,1.001457
admin_overhead_ratio,1.032766e-17,1.001457
net_asset_margin,-7.745742e-18,1.001457
labor_cost_ratio,-3.614680e-17,1.001457
solvency_ratio,1.420053e-17,1.001457
revenue_yoy_growth,1.032766e-17,1.001457


Write feature_matrix.parquet

In [12]:
MATRIX_PATH = DATA_DIR / "feature_matrix.parquet"
feature_matrix.to_parquet(MATRIX_PATH, index=False, engine="pyarrow")

print(f"feature_matrix.parquet written: {feature_matrix.shape[0]} orgs, {len(FEATURE_COLS)} features")
print(f"Path: {MATRIX_PATH.resolve()}")
print(f"\nFinal feature columns feeding XGBoost + FAISS:")
for col in FEATURE_COLS:
    print(f"  {col}")

feature_matrix.parquet written: 344 orgs, 6 features
Path: C:\Users\Edrill-LT\Documents\Projects\Atlas990\backend\app\data\processed\feature_matrix.parquet

Final feature columns feeding XGBoost + FAISS:
  program_expense_ratio
  admin_overhead_ratio
  net_asset_margin
  labor_cost_ratio
  solvency_ratio
  revenue_yoy_growth
